In [49]:
2+2


4

In [50]:
from langchain_openai import ChatOpenAI
from langchain_community.tools  import WikipediaQueryRun
from dotenv import load_dotenv
from langchain_community.utilities import WikipediaAPIWrapper

load_dotenv()

True

In [51]:
llm =ChatOpenAI(model="gpt-4o-mini",temperature=0.5)


In [52]:
wiki_wrapper=WikipediaAPIWrapper(top_k_results=3,doc_content_chars_max=400)
wiki_tool=WikipediaQueryRun(api_wrapper=wiki_wrapper,description="query wikipedia")



In [53]:
wiki_tool.name

'wikipedia'

In [54]:
wiki_tool.invoke("what is genai")


'Page: Generative artificial intelligence\nSummary: Generative artificial intelligence (Generative AI or GenAI) is a subfield of artificial intelligence that uses generative models to generate text, images, videos, audio, software code or other forms of data. These models learn the underlying patterns and structures of their training data and use them to produce new data in response to input, which '

In [55]:
from langchain_community.tools.tavily_search import TavilySearchResults

tavily_tool=TavilySearchResults()

result=tavily_tool.invoke({"query": "what is genai"})
print(type(result))
r = result[0]
print(r["title"])
print(r["url"])
print(r["content"])

<class 'list'>
Generative AI (GenAI): Definition, Importance, and Applications
https://www.denodo.com/en/glossary/generative-ai-definition-importance-applications
## How GenAI Works

GenAI leverages deep learning models trained on vast datasets to generate new outputs. The most common approaches include: [...] Generative AI (GenAI) refers to a class of artificial intelligence (AI) models that create new content, such as text, images, audio, and video, based on existing data. Using advanced deep learning techniques like generative adversarial networks (GANs) and transformers, GenAI applications can produce human-like responses, realistic visuals, and creative compositions.

## Why Is GenAI Important? [...] GenAI is important because it is transforming a wide variety of industries by automating content creation, enhancing personalization, and even driving innovation. Key benefits include:


In [56]:
def add_tool(a:int,b:int)->int:
    """This tool accepts two paramerters of type int for addition
    and returns the output as sum of two numbers in int"""
    return a+b

In [57]:
tools=[tavily_tool,add_tool]
llm_with_tools=llm.bind_tools(tools=tools)

llm_with_tools.invoke("what is genAI")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 127, 'total_tokens': 148, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_29330a9688', 'id': 'chatcmpl-CvglpZOsrM5VwPq7hkduIllNitN5W', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b9cf9-4476-7c53-bfa8-0ed67365db8d-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'what is genAI'}, 'id': 'call_QqQPXueGzzD8dzZjPh0tAH28', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 127, 'output_tokens': 21, 'total_tokens': 148, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reas

In [58]:
from langgraph.graph import StateGraph,START,END
from langgraph.prebuilt import ToolNode,tools_condition
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages
from typing import TypedDict,Annotated
from langgraph.checkpoint.memory import InMemorySaver



In [59]:
class State(TypedDict):
    messages:Annotated[list[AnyMessage],add_messages]

def llm_calling(state:State):
    return {
        "messages":[llm_with_tools.invoke(state["messages"])]
    }


In [60]:
graph=StateGraph(State)

graph.add_node("llm_calling",llm_calling)
graph.add_node("tools",ToolNode(tools=tools,messages_key="messages"))

graph.add_edge(START,"llm_calling")
graph.add_conditional_edges("llm_calling",tools_condition)
graph.add_edge("tools","llm_calling")
graph.add_edge("llm_calling",END)

memory=InMemorySaver()
workflow=graph.compile(checkpointer=memory)
config={"configurable":{"thread_id":2}}

print(workflow.get_graph().draw_mermaid())



---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	llm_calling(llm_calling)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> llm_calling;
	llm_calling -.-> __end__;
	llm_calling -.-> tools;
	tools --> llm_calling;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [62]:
response=workflow.invoke({"messages":"what is GENAI"},config=config,stream_mode="values")

for r in response["messages"]:
    r.pretty_print()

================================ Human Message =================================

what is the sum of 3 and 5
================================== Ai Message ==================================
Tool Calls:
  add_tool (call_HbFVoXTmhjJKa6L6MYLlYRCc)
 Call ID: call_HbFVoXTmhjJKa6L6MYLlYRCc
  Args:
    a: 3
    b: 5
================================= Tool Message =================================
Name: add_tool

8
================================== Ai Message ==================================

The sum of 3 and 5 is 8.
================================ Human Message =================================

what is GENAI
================================== Ai Message ==================================
Tool Calls:
  tavily_search_results_json (call_tzGBDF7GbXi6m2pM6LZg7Sr3)
 Call ID: call_tzGBDF7GbXi6m2pM6LZg7Sr3
  Args:
    query: GENAI
================================= Tool Message =================================
Name: tavily_search_results_json

[{"title": "What is Gen AI? - GenAI for Students - UC